# Occam's Folly — WC 2026 Prediction Model\n\nThis notebook walks through the full prediction pipeline for the **Rotate 2026 World Cup prediction game**.\n\nRun cells top to bottom. Each cell builds on the previous one.

---
## Step 1 — Install & Setup

In [ ]:
import subprocess, sys, os

project_root = os.path.abspath('..')

print(f'Python: {sys.version}')
assert sys.version_info >= (3, 9), 'Requires Python 3.9+'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                os.path.join(project_root, 'requirements.txt'), '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', project_root, '-q'], check=True)

sys.path.insert(0, os.path.join(project_root, 'scripts'))

import warnings
warnings.filterwarnings('ignore')
print('Setup complete.')

---
## Step 2 — Train the Model

Loads ~47,000 international matches (2010–2026), builds 120 features across 13 modules, then trains:
- **XGBoost** 3-class classifier + Temperature Scaling calibration
- **Bayesian Hierarchical Poisson** (MAP, Dixon-Coles ρ, half-life from Kalman EM q)
- **Context-Adaptive Ensemble** (per-match sigmoid weight on odds/uncertainty/H2H)

Takes ~60 seconds.

In [ ]:
from predict_wc2026 import train_model

xgb, temp_cal, bp, ensemble, all_data, train_df = train_model(quiet=False)

---
## Step 3 — Predict All 72 Group Stage Fixtures

In [ ]:
from predict_wc2026 import predict_group_stage, load_actual_results, merge_actual_results
import pandas as pd

match_data = predict_group_stage(xgb, temp_cal, bp, ensemble, all_data, quiet=True)
actual = load_actual_results()
match_data = merge_actual_results(match_data, actual)

rows = [{
    'Group': m['group'],
    'Match': f"{m['home_team']} vs {m['away_team']}",
    'Home win': f"{m['p_home']:.1%}",
    'Draw': f"{m['p_draw']:.1%}",
    'Away win': f"{m['p_away']:.1%}",
    'Played': '✓' if m.get('played') else ''
} for m in match_data]

pd.DataFrame(rows)

---
## Step 4 — Generate the Competition Submission

Runs 30,000 global Monte Carlo simulations across all 12 groups to estimate P(advance to R32),
correctly applying the best-8 third-place rule. Then picks the scoreline per match that
**maximises expected competition points** under the 5/3/2/0 scoring rubric.

Saves to `output/output.csv`.

In [ ]:
import subprocess, os
result = subprocess.run(
    [sys.executable, os.path.join(project_root, 'scripts', 'generate_submission.py')],
    capture_output=True, text=True, cwd=project_root
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

In [ ]:
submission = pd.read_csv(os.path.join(project_root, 'output', 'output.csv'))
print(f'{len(submission)} predictions')
submission

---
## Step 5 — Recording Actual Results (as the tournament plays out)

After each match, record the actual score. The model will lock that match to the real result,
update the Kalman EKF with the new goals data, and re-simulate only the remaining fixtures.

In [ ]:
# Example: record Mexico 2-1 South Africa, Group A
# Uncomment and edit to record a real result

# subprocess.run([
#     sys.executable,
#     os.path.join(project_root, 'scripts', 'update_wc2026.py'),
#     '--result', 'Mexico vs South Africa',
#     '--score', '2-1',
#     '--group', 'A'
# ], check=True)

# Then re-run Steps 3 & 4 above to refresh predictions.

# List all recorded results so far:
import json
results_file = os.path.join(project_root, 'data', 'wc2026_actual_results.json')
with open(results_file) as f:
    recorded = json.load(f)
print(f'{len(recorded)} results recorded so far.')
if recorded:
    pd.DataFrame(recorded)

---
## Model Architecture Summary

| Layer | What it does |
|---|---|
| **XGBoost** | 3-class classifier over ~120 features; match-importance weighted |
| **Temperature Scaling** | Single scalar T calibrates XGBoost log-probabilities (Guo et al. 2017) |
| **Bayesian Poisson** | MAP log-linear goal model; Dixon-Coles ρ; half-life from Kalman EM q |
| **Context-Adaptive Ensemble** | Per-match α = sigmoid(w · [odds_avail, kalman_unc, log1p_h2h]) |
| **WC 2026 Post-Processing** | Venue λ adjust · sofifa quality nudge · player absence penalty |

| Feature module | Signal |
|---|---|
| Kalman EKF | Time-varying attack/defence; EM-tuned process noise q; forward-only causal states |
| Glicko-2 | Rating μ + deviation φ + volatility σ; Illinois update; match-importance weighted |
| Elo | Classic Elo; K scaled by match importance |
| Form | Rolling pts/goals/GD over 5/10/20 matches |
| Strength of schedule | Opponent-quality-adjusted win rate |
| Head-to-head | H2H win rate, avg goals, last 10 meetings |
| xG form | Rolling xG/xGA (UEFA/AFC/CONMEBOL qualifiers) |
| Transfermarkt | Squad market values June 2026 |
| FIFA Rankings | Official ranking points |
| Confederation | Data-derived offsets: CONMEBOL=65, UEFA=50, AFC=25, CAF=10, CONCACAF=−20 |
| Odds | Bookmaker closing implied probs; drives context-adaptive ensemble weight |

## Auto-Tuning (runs automatically on first use)

Two hyperparameters are tuned once and cached — no flags needed:

| Parameter | Method | Cache file |
|---|---|---|
| XGBoost hyperparameters | Optuna TPE, 60 trials — WC 2018 & 2022 group stage as val folds | `data/xgb_tuned_params.json` |
| Friendly match weight | Grid search [0.05…1.0] — minimises log-loss on WC 2018 & 2022 | `data/tuned_params.json` |

Run `python3 scripts/pipeline.py` once to tune and cache. All subsequent runs (including `generate_submission.py`) load from cache automatically. Force re-tune with `--tune` (XGBoost) or `--retune` (friendly weight).